# Stage1 joint cash-and-cap 0bp local value
One frozen 2018-2022 training label pass, no model or account.


In [1]:
import copy,datetime,hashlib,json,os,sys,shlex,tempfile
from pathlib import Path
root=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parents[1];sys.path.insert(0,str(root))
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from src.backtest.ibkr_tiered import TieredFees,settle_tiered_event
from src.data.nas_market_data import archive_nas_recorder,materialize_nas_evidence,_ssh
card=root/'research/stage1_joint_cash_cap_soxx_0bp_local_value_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists()
for p,k in [(root/src['training_reference_card']/'summary.json','training_reference_summary_sha256'),(root/src['fee_plan'],'fee_plan_sha256'),(root/'research/stage1_original_0bp_cash_layer_window_bound_20261023/summary.json','cash_layer_diagnosis_summary_sha256'),(root/'research/stage1_cash_drag_overlay_20261010/summary.json','closed_old_cash_overlay_summary_sha256'),(root/'src/backtest/ibkr_tiered.py','fee_module_sha256'),(root/'research/stage1_0946_peer_breadth_source_20261023/summary.json','future_0946_peer_breadth_source_summary_sha256')]:assert sha(p)==src[k],str(p)
source_summary=json.loads((root/src['training_reference_card']/'summary.json').read_text());assert source_summary['recorder_id']==src['training_reference_recorder_id'] and source_summary['experiment_id']==src['training_reference_experiment_id'] and source_summary['nas_readback']['verified']
fee_plan=json.loads((root/src['fee_plan']).read_text());provider=(root/'data/qlib/full_path_tiered_allocation_20261013').resolve();tracking=(root/src['training_reference_card']/'qlib_recorder').resolve();os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(tracking),'default_exp_name':Path(src['training_reference_card']).name}})
ref=R.get_recorder(experiment_name=Path(src['training_reference_card']).name,recorder_id=src['training_reference_recorder_id']);frame=ref.load_object('inputs/source_frame').reset_index(drop=True);bundle=ref.load_object('paths/reference_0');native=ref.load_object('native/reference_0');action=ref.load_object('inputs/action_plan');path=bundle['replay'].reset_index(drop=True);targets=bundle['targets'].reset_index(drop=True)
assert native['audit']['status']=='passed' and native['audit']['slippage_bps']==0 and len(frame)==len(path)==len(targets)==1061
assert frame.trade_date.equals(path.trade_date) and frame.trade_date.equals(targets.trade_date)
cal=list(pd.to_datetime(D.calendar(start_time='2018-10-15',end_time='2022-12-30',freq='day')).date);assert cal==list(pd.to_datetime(frame.trade_date).dt.date)
assert action['recovery_spec']['minimum_cash_weight']==.15 and action['recovery_spec']['maximum_soxl_weight']==.30
assert (pd.to_datetime(frame.feature_available_at,utc=True)<=pd.to_datetime(frame.decision_at,utc=True)).all()
state=card/'run_state.json';state.write_text(json.dumps({'status':'started','plan_sha256':sha(card/'plan.json'),'label_passes_started':1},indent=2)+'\n')
fees=TieredFees(fee_plan['tariff'],slippage_bps=0);q=np.zeros(2);cash=100000.;records=[];max_ledger_error=0.;max_branch_error=0.;whole_share_changed=0
for i,row in frame.iterrows():
 saved=path.iloc[i];target=targets.iloc[i][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float);raw=row[['soxl_weight','soxx_weight']].to_numpy(float);cap=float(saved.current_beta_cap)
 raw_beta=float(3*raw[0]+raw[1]);base_beta=float(3*target[0]+target[1]);binding=bool(raw_beta>cap+1e-9);ceiling=cap+(.10 if binding else 0.);delta=max(0.,min(.10,max(0.,target[2]-.15),max(0.,ceiling-base_beta))) if bool(saved.execute_open_target) else 0.
 eligible=bool(saved.execute_open_target and delta>1e-10);mature=eligible and i+1<len(frame)
 candidate_target=target+np.array([0.,delta,-delta]);assert candidate_target.min()>=-1e-12 and abs(candidate_target.sum()-1)<1e-9 and candidate_target[2]>=.15-1e-10 and 3*candidate_target[0]+candidate_target[1]<=ceiling+1e-9
 today=row[['soxl_open','soxx_open']].to_numpy(float);raw_today=row[['raw_soxl_open','raw_soxx_open']].to_numpy(float);stamp=pd.Timestamp(row.trade_date)+pd.Timedelta(hours=9,minutes=47);nav=float(q@today+cash)
 label=np.nan;available=pd.NaT
 if mature:
  next_row=frame.iloc[i+1];reset=targets.iloc[i+1][['soxl_weight','soxx_weight','cash_weight']].to_numpy(float)
  fee_a=copy.deepcopy(fees);fee_b=copy.deepcopy(fees);assert fee_a.monthly_shares==fee_b.monthly_shares==fees.monthly_shares
  a=settle_tiered_event(q,cash,desired=target,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fee_a)
  b=settle_tiered_event(q,cash,desired=candidate_target,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fee_b)
  assert np.isclose(a['nav_before'],b['nav_before'],rtol=0,atol=1e-8) and b['cash']>=-1e-7
  whole_share_changed+=int(np.max(np.abs(a['quantities']-b['quantities'])*today)/nav>1e-8)
  tomorrow=next_row[['soxl_open','soxx_open']].to_numpy(float);raw_tomorrow=next_row[['raw_soxl_open','raw_soxx_open']].to_numpy(float);nextstamp=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=47)
  ar=settle_tiered_event(a['quantities'],a['cash'],desired=reset,adjusted_prices=tomorrow,raw_prices=raw_tomorrow,stamp=nextstamp,fees=fee_a)
  br=settle_tiered_event(b['quantities'],b['cash'],desired=reset,adjusted_prices=tomorrow,raw_prices=raw_tomorrow,stamp=nextstamp,fees=fee_b)
  av=float(ar['quantities']@tomorrow+ar['cash']);bv=float(br['quantities']@tomorrow+br['cash']);assert np.isfinite([av,bv]).all() and min(av,bv)>0
  label=(bv-av)/nav;available=pd.Timestamp(next_row.trade_date)+pd.Timedelta(hours=9,minutes=49)
 control=settle_tiered_event(q,cash,desired=target if bool(saved.execute_open_target) else None,adjusted_prices=today,raw_prices=raw_today,stamp=stamp,fees=fees)
 if mature:
  err=float(np.max(np.abs(np.r_[a['quantities'],a['cash']]-np.r_[control['quantities'],control['cash']])));max_branch_error=max(max_branch_error,err);assert err<=1e-8
 err=float(np.max(np.abs(np.r_[control['quantities'],control['cash']]-saved[['ending_soxl_quantity','ending_soxx_quantity','ending_cash']].to_numpy(float))));max_ledger_error=max(max_ledger_error,err);assert err<=1e-6,(i,err)
 records.append({'signal_date':row.signal_date,'trade_date':row.trade_date,'decision_at':row.decision_at,'eligible':eligible,'cap_binding':binding,'delta_target_beta':delta,'label_available_at':available,'joint_local_fee_net_advantage':label})
 q,cash=control['quantities'],control['cash']
labels=pd.DataFrame(records);valid=labels[labels.joint_local_fee_net_advantage.notna()];assert len(labels)==1061 and len(valid)>0 and (pd.to_datetime(valid.label_available_at)>pd.to_datetime(valid.decision_at)).all()
annual=[]
for year,g in labels.groupby(pd.to_datetime(labels.trade_date).dt.year):
 z=g.joint_local_fee_net_advantage.dropna();annual.append({'year':int(year),'sessions':len(g),'eligible':int(g.eligible.sum()),'mature':len(z),'binding_mature':int((g.cap_binding & g.joint_local_fee_net_advantage.notna()).sum()),'nonbinding_mature':int((~g.cap_binding & g.joint_local_fee_net_advantage.notna()).sum()),'positive':int((z>0).sum()),'negative':int((z<0).sum()),'mean':float(z.mean()) if len(z) else None})
by_layer={}
for name,g in [('binding',valid[valid.cap_binding]),('nonbinding',valid[~valid.cap_binding])]:by_layer[name]={'mature':len(g),'mean':float(g.joint_local_fee_net_advantage.mean()) if len(g) else None,'positive':int((g.joint_local_fee_net_advantage>0).sum()),'negative':int((g.joint_local_fee_net_advantage<0).sum())}
result={'decision':'joint_cash_cap_training_local_value_only_no_fit_or_account','sessions':len(labels),'eligible':int(labels.eligible.sum()),'mature':len(valid),'whole_share_changed_mature':whole_share_changed,'mean_local_advantage':float(valid.joint_local_fee_net_advantage.mean()),'median_local_advantage':float(valid.joint_local_fee_net_advantage.median()),'positive':int((valid.joint_local_fee_net_advantage>0).sum()),'negative':int((valid.joint_local_fee_net_advantage<0).sum()),'annual':annual,'by_layer':by_layer,'max_control_cash_share_ledger_error_usd':max_ledger_error,'max_branch_control_identity_error_usd':max_branch_error,'last_label_unknown':str(labels.trade_date.iloc[-1].date()),'new_market_requests':0,'new_model_fits':0,'new_native_accounts':0,'label_passes':1,'historical_independence':False,'interpretation':'2018-2022 original-architecture OOF local 0bp Tiered target only. A one-step common reset does not imply a full Stage1 own-state policy, 2023+ SOXX account or independent forward evidence.'}
(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan);rec.save_objects(artifact_path='analysis',labels=labels,annual=annual,result=result);rec.log_metrics(mature_rows=len(valid),binding_rows=by_layer['binding']['mature'],nonbinding_rows=by_layer['nonbinding']['mature'],mean_local_advantage=result['mean_local_advantage'],new_model_fits=0,new_native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'source_recorder_id':src['training_reference_recorder_id'],'new_model_fits':0,'new_native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['analysis/labels','analysis/annual','analysis/result']}}
binding=json.loads((root/src['training_reference_card']/'nas_binding.json').read_text());binding.pop('evidence',None);binding.update(source_id=plan['experiment'],source_plan='plan.json',as_of='2022-12-30');(card/'nas_binding.json').write_text(json.dumps(binding,ensure_ascii=False,indent=2)+'\n')
evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);remote=_ssh(binding['ssh'],'sha256sum '+shlex.quote(evidence['nas_path'])).split()[0];assert remote==evidence['sha256']
with tempfile.TemporaryDirectory(prefix='soxl-joint-readback-') as temp:
 restored=Path(temp)/'card';restored.mkdir();materialize_nas_evidence(card/'nas_binding.json',card=restored);rel=Path('qlib_recorder')/eid/rid/'artifacts/analysis/labels';assert sha(card/rel)==sha(restored/rel)
summary['nas_evidence']=evidence;summary['nas_readback']={'verified':True,'sha256':remote,'scoped_restored_labels_sha256':sha(card/rel)}
(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n');state.write_text(json.dumps({'status':'completed','plan_sha256':sha(card/'plan.json'),'label_passes_completed':1,'recorder_id':rid},indent=2)+'\n')
print(json.dumps({'result':result,'recorder_id':rid,'NAS_sha256':remote},ensure_ascii=False))

{"result": {"decision": "joint_cash_cap_training_local_value_only_no_fit_or_account", "sessions": 1061, "eligible": 689, "mature": 688, "whole_share_changed_mature": 669, "mean_local_advantage": 8.434237351493289e-05, "median_local_advantage": 3.2004558759215424e-05, "positive": 358, "negative": 311, "annual": [{"year": 2018, "sessions": 53, "eligible": 19, "mature": 19, "binding_mature": 18, "nonbinding_mature": 1, "positive": 11, "negative": 8, "mean": 8.497426904421524e-05}, {"year": 2019, "sessions": 252, "eligible": 138, "mature": 138, "binding_mature": 39, "nonbinding_mature": 99, "positive": 68, "negative": 55, "mean": 4.16109415670285e-05}, {"year": 2020, "sessions": 253, "eligible": 202, "mature": 202, "binding_mature": 182, "nonbinding_mature": 20, "positive": 123, "negative": 79, "mean": 0.0002442638818866019}, {"year": 2021, "sessions": 252, "eligible": 82, "mature": 82, "binding_mature": 56, "nonbinding_mature": 26, "positive": 43, "negative": 35, "mean": 0.000380017502993

# Stage1 联合现金与风险上限：0bp 局部价值

## 结论

冻结的 2018–2022 年原架构 OOF 训练账本完成一次 Qlib 标签计算：1061 个交易日中，689 日满足执行条件，688 个标签在下一交易日成熟。候选只从现金向 SOXX 增加最多 0.10 权重；风险上限受限时至多额外放宽 0.10 beta，其他日使用原有剩余上限。两条分支沿用同一初始资金、09:47 整股成交、IBKR Tiered 手续费及下一交易日共同目标复位，附加滑点为 0bp。原控制分支的现金和持仓账本误差为零。没有模型拟合或完整策略账户。

整体平均局部净值优势为 +0.00008434，其中风险上限受限的 540 日为 +0.00009535，未受限的 148 日为 +0.00004418；只有 669 个成熟日实际改变整股持仓。2022 年 247 个成熟日的平均优势为 −0.00012078。未受限日按年为 1、99、20、26、2 日；2022 年仅 2 日，无法据此检验该层在不同年份的稳定性。正均值来自已复用的训练历史，且局部共同复位标签不等于连续持仓策略收益。

**固定版本停在标签和支持度阶段。** 不把 09:46 同行广度接到该标签上拟合，也不启动 2023 年以后的 Stage1 完整账户：现有未受限层几乎没有 2022 年检验样本，而 2022 年整体局部收益转负。下一机制需要先解释这种跨年差异，并在拟合前冻结接收器、可用时点、分年支持和信息门。Stage1 f696 分配器与 b472 行动、风控、恢复链仍是完整账户的唯一父本；未来先以 0bp 加 Tiered 手续费对 SOXX 全历史硬门检验，通过后才谈 2/5bp 和独立前瞻证据。本卡没有证明超过 SOXX，也不是合格前瞻交易日。

证据：`plan.json`、`result.json`、`summary.json`、Qlib Recorder `dd91c4770a354dfe8dc275774e249020`；NAS 原件与定向恢复校验见 `nas_binding.json`。
